In [1]:
%pwd

'd:\\Tipto\\OmniChef-Nexus\\notebooks'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'd:\\Tipto\\OmniChef-Nexus'

In [4]:
# load the 15k recipe file
import pandas as pd 

df = pd.read_csv("data/all csv files/recipes_15k_samples.csv")
df.shape

(15698, 15)

In [5]:
df.head()

,name,minutes,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,recipe_id,rating,review,num of ratings,markdown_recipe,image_path
0,chicken lickin good pork chops,500,"['weeknight', 'time-to-make', 'course', 'main-...","[105.7, 8.0, 0.0, 26.0, 5.0, 4.0, 3.0]",5,"['dredge pork chops in mixture of flour , salt...",here's and old standby i enjoy from time to ti...,"['lean pork chops', 'flour', 'salt', 'dry must...",7,63986,4.368421,"[""I made this for dinner tonight and the chops...",19.0,# Chicken Lickin Good Pork Chops\n**Recipe I...,data/output/images/recipe_id_63986.png
1,chile rellenos,45,"['60-minutes-or-less', 'time-to-make', 'course...","[94.0, 10.0, 0.0, 11.0, 11.0, 21.0, 0.0]",9,"['drain green chiles', 'sprinkle cornstarch on...",a favorite from a local restaurant no longer i...,"['egg roll wrap', 'whole green chilies', 'chee...",5,43026,4.045455,"['Grandma Pam, Oh my goodness,these were so ea...",22.0,# Chile Rellenos\n**Recipe ID:** 43026\n**Cook...,data/output/images/recipe_id_43026.png
2,chinese candy,15,"['15-minutes-or-less', 'time-to-make', 'course...","[232.7, 21.0, 77.0, 4.0, 6.0, 38.0, 8.0]",4,['melt butterscotch chips in heavy saucepan ov...,"a little different, and oh so good. i include ...","['butterscotch chips', 'chinese noodles', 'sal...",3,23933,4.833333,"[""A bit different, maybe...but these are delic...",12.0,# Chinese Candy\n**Recipe ID:** 23933\n**Cook...,data/output/images/recipe_id_23933.png
3,grilled venison burgers,26,"['30-minutes-or-less', 'time-to-make', 'course...","[190.9, 10.0, 10.0, 10.0, 45.0, 15.0, 2.0]",13,"['in bowl , mix dry ingredients', 'add venison...",delicious venison burgers with that,"['ground venison', 'egg substitute', 'non-fat ...",10,54100,4.428571,"[""I really enjoyed these burgers. The taste wa...",14.0,# Grilled Venison Burgers\n**Recipe ID:** 541...,data/output/images/recipe_id_54100.png
4,healthy for them yogurt popsicles,10,"['15-minutes-or-less', 'time-to-make', 'course...","[164.6, 3.0, 5.0, 1.0, 4.0, 6.0, 11.0]",3,"['mix all the ingredients using a blender', 'p...",my children and their friends ask for my homem...,"['milk', 'frozen juice concentrate', 'plain yo...",3,67664,4.833333,['My lips are still smacking away. These are F...,12.0,# Healthy For Them Yogurt Popsicles\n**Recipe...,data/output/images/recipe_id_67664.png


In [6]:
for col in df.columns.to_list():
    print(f"{col} => {type(col)}")

name => <class 'str'>
minutes => <class 'str'>
tags => <class 'str'>
nutrition => <class 'str'>
n_steps => <class 'str'>
steps => <class 'str'>
description => <class 'str'>
ingredients => <class 'str'>
n_ingredients => <class 'str'>
recipe_id => <class 'str'>
rating => <class 'str'>
review => <class 'str'>
num of ratings => <class 'str'>
markdown_recipe => <class 'str'>
image_path => <class 'str'>


In [7]:
import ast 
import json

list_columns = ['tags', 'nutrition', 'steps', 'ingredients' , 'review']
for col in list_columns:
    df[col] = df[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x , str) else x
    )

In [8]:
# renaming image_path to file name so that hf ensure image preview
df = df.rename(columns = {'image_path' : 'file_name'})

In [9]:
df.head(2)

,name,minutes,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,recipe_id,rating,review,num of ratings,markdown_recipe,file_name
0,chicken lickin good pork chops,500,"[weeknight, time-to-make, course, main-ingredi...","[105.7, 8.0, 0.0, 26.0, 5.0, 4.0, 3.0]",5,"[dredge pork chops in mixture of flour , salt ...",here's and old standby i enjoy from time to ti...,"[lean pork chops, flour, salt, dry mustard, ga...",7,63986,4.368421,[I made this for dinner tonight and the chops ...,19.0,# Chicken Lickin Good Pork Chops\n**Recipe I...,data/output/images/recipe_id_63986.png
1,chile rellenos,45,"[60-minutes-or-less, time-to-make, course, mai...","[94.0, 10.0, 0.0, 11.0, 11.0, 21.0, 0.0]",9,"[drain green chiles, sprinkle cornstarch on sh...",a favorite from a local restaurant no longer i...,"[egg roll wrap, whole green chilies, cheese, c...",5,43026,4.045455,"[Grandma Pam, Oh my goodness,these were so eas...",22.0,# Chile Rellenos\n**Recipe ID:** 43026\n**Cook...,data/output/images/recipe_id_43026.png


### Prepare dataset for upload to Hugging Face

For uploading dataset to we need:
1. Folder of Images
2. `jsonl` file in the folder with `file_name` field point

In [10]:
OUTPUT_DIR  = "data/output/images"   
OUTPUT_JSONL = os.path.join(OUTPUT_DIR, "metadata.jsonl")

In [11]:
from pathlib import Path

count = 0
for _ , row in df.iterrows():
    count += 1
    
    file_path = row.get("file_name")
    file_name = Path(str(file_path)).name 
    full_image_path = Path(OUTPUT_DIR)/file_name
    full_image_path = full_image_path.as_posix()
    print(full_image_path)
    if count == 5:
        break

data/output/images/recipe_id_63986.png
data/output/images/recipe_id_43026.png
data/output/images/recipe_id_23933.png
data/output/images/recipe_id_54100.png
data/output/images/recipe_id_67664.png


In [12]:
written = 0
skipped = 0

with open(OUTPUT_JSONL , "w" , encoding = 'utf-8') as f:
    for _ , row in df.iterrows():
        file_path = row.get("file_name" , "")
        file_name = Path(str(file_path)).name 
        full_image_path = Path(OUTPUT_DIR)/file_name
        full_image_path = full_image_path.as_posix()
        
        # skip rows whose file doesn't exists
        if not os.path.exists(full_image_path):
            skipped += 1
            continue
        
        record = {
            "file_name" : file_name,
            "name" : row.get("name"),
            "minutes" : int(row["minutes"]) if pd.notna(row.get("minutes")) else None,
            "tags" : row.get("tags"),
            "nutrition" : row.get("nutrition"),
            "n_steps" : int(row["n_steps"]) if pd.notna(row.get("n_steps")) else None,
            "steps" : row.get("steps"),
            "description" : row.get("description"),
            "ingredients" : row.get("ingredients"),
            "n_ingredients" : int(row["n_ingredients"]) if pd.notna(row.get("n_ingredients")) else None,
            "recipe_id" : int(row["recipe_id"]) if pd.notna(row.get("recipe_id")) else None,
            "rating" : float(row["rating"]) if pd.notna(row.get("rating")) else None,
            "review" : row.get("review"),
            "num_of_ratings" : float(row["num_of_ratings"]) if pd.notna(row.get("num_of_ratings")) else None,
            "markdown_recipe": row.get("markdown_recipe"),
        }
        
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        written += 1
        

print(f"written : {written:,} rows")
print(f"skipped : {skipped:,} rows (image file missing)")
print(f"Output  : {OUTPUT_JSONL}")     

written : 15,698 rows
skipped : 0 rows (image file missing)
Output  : data/output/images\metadata.jsonl


* created the repo on hugging face.

In [13]:
from huggingface_hub import HfApi
from dotenv import load_dotenv

load_dotenv()

REPO_ID = "tiptoghosh/food-recipes-15k"
IMAGE_DIR = "data/output/images"
HF_TOKEN = os.getenv("HF_TOKEN")

In [14]:
api = HfApi(token = HF_TOKEN)

In [15]:
print(f"Uploading folder to: {REPO_ID}")
print(f"Source folder : {IMAGE_DIR}")
print(f"Total files: {len(os.listdir(IMAGE_DIR)):,}\n")

Uploading folder to: tiptoghosh/food-recipes-15k
Source folder : data/output/images
Total files: 15,700



In [ ]:
api.upload_large_folder(
    repo_id = REPO_ID,
    folder_path = IMAGE_DIR,
    repo_type = 'dataset'
)

Folder 'root' contains 15,699 entries (15,699 files and 0 subdirectories). This exceeds the recommended 10,000 entries per folder.
Consider reorganising into sub-folders.


Recovering from metadata files:   0%|          | 0/15699 [00:00<?, ?it/s]




---------- 2026-04-16 21:56:48 (0:00:00) ----------
Files:   hashed 15698/15699 (7.5G/7.7G) | pre-uploaded: 0/15698 (0.0/7.7G) (+1 unsure) | committed: 0/15699 (0.0/7.7G) | ignored: 0
Workers: hashing: 1 | get upload mode: 0 | pre-uploading: 7 | committing: 0 | waiting: 0
---------------------------------------------------


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


---------- 2026-04-16 22:01:48 (0:05:00) ----------
Files:   hashed 15699/15699 (7.7G/7.7G) | pre-uploaded: 256/15699 (124.4M/7.7G) | committed: 175/15699 (85.0M/7.7G) | ignored: 0
Workers: hashing: 0 | get upload mode: 0 | pre-uploading: 8 | committing: 0 | waiting: 0
---------------------------------------------------
                       
---------- 2026-04-16 22:02:48 (0:06:00) ----------
Files:   hashed 15699/15699 (7.7G/7.7G) | pre-uploaded: 256/15699 (124.4M/7.7G) | committed: 175/15699 (85.0M/7.7G) | ignored: 0
Workers: hashing: 0 | get upload mode: 0 | pre-uploading: 8 | committing: 0 | waiting: 0
---------------------------------------------------
                       
---------- 2026-04-16 22:03:48 (0:07:00) ----------
Files:   hashed 15699/15699 (7.7G/7.7G) | pre-uploaded: 256/15699 (124.4M/7.7G) | committed: 175/15699 (85.0M/7.7G) | ignored: 0
Workers: hashing: 0 | get upload mode: 0 | pre-uploading: 8 | committing: 0 | waiting: 0
-------------------------------------

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "imagefolder",
    data_dir = IMAGE_DIR,
    split = 'train'
)

Resolving data files:   0%|          | 0/15699 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['image', 'name', 'minutes', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients', 'recipe_id', 'rating', 'review', 'num_of_ratings', 'markdown_recipe'],
    num_rows: 15698
})

In [ ]:
dataset.push_to_hub(
    repo_id = REPO_ID,
    token = HF_TOKEN,
    max_shard_size = "500MB"
)

Uploading the dataset shards:   0%|          | 0/16 [00:00<?, ? shards/s]

Map:   0%|          | 0/982 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            